# Lab 04 — One-Time Structure Setup

Run this notebook **manually** when the Lab 4 environment must be created or verified.

This notebook is intentionally **not part of the production Databricks Job**.

It performs structural setup only:
- creates the Unity Catalog catalog and schema when permitted;
- creates the configured managed or external volume;
- creates the Lab 4 folder structure;
- validates that the volume is accessible.

Runtime configuration lives in `lab04_00_config`.


In [0]:
%run ./lab04_00_config


In [0]:
# Structural DDL belongs here, outside the production Job.

spark.sql(f"CREATE CATALOG IF NOT EXISTS `{catalog}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")

if volume_type == "external":
    spark.sql(
        f"""
        CREATE EXTERNAL VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{volume_name}`
        LOCATION '{external_volume_url}'
        """
    )
else:
    spark.sql(
        f"CREATE VOLUME IF NOT EXISTS `{catalog}`.`{schema}`.`{volume_name}`"
    )

print(f"Catalog ready: {catalog}")
print(f"Schema ready: {catalog}.{schema}")
print(f"Volume ready: {catalog}.{schema}.{volume_name} ({volume_type})")
if volume_type == "external":
    print(f"External volume location: {external_volume_url}")


In [0]:
# Folder creation is also kept outside the production Job.
for path in paths.values():
    dbutils.fs.mkdirs(path)

print(f"Created or verified {len(paths)} Lab 4 folders under {volume_root}")
for purpose, path in paths.items():
    print(f"  {purpose}: {path}")


In [0]:
# Permanent Delta table structures used by production Job tasks.
# All CREATE/DROP statements live here so pipeline notebooks remain DDL-free.

if reset_demo_objects and batch_id != "initial":
    raise ValueError(
        "reset_demo_objects=true is allowed only with batch_id=initial."
    )

permanent_tables = [
    table_names["product_scd2"],
    table_names["product_scd1"],
    table_names["silver_transactions"],
    table_names["quality_metrics"],
    table_names["quarantine"],
    table_names["bronze"],
]

if reset_demo_objects:
    for table_name in permanent_tables:
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")
        print(f"Dropped for clean manual rebuild: {table_name}")

# ---------------- Bronze ----------------
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {table_names["bronze"]} (
        InvoiceNo STRING,
        StockCode STRING,
        Description STRING,
        Quantity BIGINT,
        InvoiceDate TIMESTAMP,
        UnitPrice DOUBLE,
        CustomerID STRING,
        Country STRING,
        _source_row_number BIGINT,
        _source_file STRING,
        _source_sheet STRING,
        _prepared_at_utc TIMESTAMP,
        _record_hash STRING,
        _input_file_path STRING,
        _input_file_name STRING,
        _input_file_size BIGINT,
        _input_file_modified_at TIMESTAMP,
        _bronze_record_id STRING,
        _batch_id STRING,
        _source_system STRING,
        _contract_version STRING,
        _bronze_ingested_at TIMESTAMP,
        _bronze_ingestion_date DATE
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'quality.layer' = 'bronze',
        'quality.source' = 'uci_online_retail'
    )
    """
)

# ---------------- Quarantine ----------------
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {table_names["quarantine"]} (
        InvoiceNo STRING,
        StockCode STRING,
        Description STRING,
        Quantity BIGINT,
        InvoiceDate TIMESTAMP,
        UnitPrice DOUBLE,
        CustomerID STRING,
        Country STRING,
        _source_row_number BIGINT,
        _source_file STRING,
        _source_sheet STRING,
        _prepared_at_utc TIMESTAMP,
        _record_hash STRING,
        _input_file_path STRING,
        _input_file_name STRING,
        _input_file_size BIGINT,
        _input_file_modified_at TIMESTAMP,
        _bronze_record_id STRING,
        _batch_id STRING,
        _source_system STRING,
        _contract_version STRING,
        _bronze_ingested_at TIMESTAMP,
        _bronze_ingestion_date DATE,
        _duplicate_rank INT,
        _quality_reasons ARRAY<STRING>,
        _quality_rule_count INT,
        _quality_status STRING,
        _quality_contract_version STRING,
        _quality_checked_at TIMESTAMP,
        _quarantine_record_id STRING,
        _quarantined_at TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'quality.layer' = 'quarantine',
        'quality.contract' = 'online_retail'
    )
    """
)

# ---------------- Quality metrics ----------------
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {table_names["quality_metrics"]} (
        bronze_rows BIGINT,
        valid_rows BIGINT,
        rejected_rows BIGINT,
        cancelled_rows BIGINT,
        missing_customer_rows BIGINT,
        duplicate_rows BIGINT,
        batch_id STRING,
        contract_version STRING,
        valid_percentage DOUBLE,
        rejected_percentage DOUBLE,
        candidate_path STRING,
        measured_at TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'quality.layer' = 'metrics',
        'quality.contract' = 'online_retail'
    )
    """
)

# ---------------- Final Silver ----------------
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {table_names["silver_transactions"]} (
        transaction_line_id STRING,
        invoice_no STRING,
        stock_code STRING,
        description STRING,
        quantity BIGINT,
        invoice_timestamp TIMESTAMP,
        unit_price DECIMAL(18,4),
        customer_id STRING,
        country STRING,
        source_record_hash STRING,
        source_batch_id STRING,
        source_file STRING,
        source_sheet STRING,
        source_row_number BIGINT,
        input_file_path STRING,
        bronze_ingested_at TIMESTAMP,
        quality_contract_version STRING,
        quality_checked_at TIMESTAMP,
        sales_amount DECIMAL(20,4),
        invoice_date DATE,
        invoice_year INT,
        invoice_month INT,
        silver_prepared_at TIMESTAMP,
        silver_created_at TIMESTAMP,
        silver_updated_at TIMESTAMP,
        silver_last_batch_id STRING
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'quality.layer' = 'silver',
        'quality.contract' = 'online_retail'
    )
    """
)

# ---------------- SCD Type 1 ----------------
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {table_names["product_scd1"]} (
        product_sk STRING,
        stock_code STRING,
        description STRING,
        latest_observed_price DECIMAL(18,4),
        source_invoice_no STRING,
        source_transaction_line_id STRING,
        source_event_timestamp TIMESTAMP,
        source_record_hash STRING,
        source_batch_id STRING,
        effective_from TIMESTAMP,
        effective_to TIMESTAMP,
        is_current BOOLEAN,
        created_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'quality.layer' = 'silver',
        'dimension.name' = 'product',
        'dimension.scd_type' = '1'
    )
    """
)

# ---------------- SCD Type 2 ----------------
spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {table_names["product_scd2"]} (
        product_version_sk STRING,
        product_sk STRING,
        stock_code STRING,
        description STRING,
        latest_observed_price DECIMAL(18,4),
        version_number INT,
        source_invoice_no STRING,
        source_transaction_line_id STRING,
        source_event_timestamp TIMESTAMP,
        source_record_hash STRING,
        source_batch_id STRING,
        effective_from TIMESTAMP,
        effective_to TIMESTAMP,
        is_current BOOLEAN,
        created_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'quality.layer' = 'silver',
        'dimension.name' = 'product',
        'dimension.scd_type' = '2'
    )
    """
)

print(
    "✅ Permanent Bronze, quarantine, quality-metrics, "
    "Silver, SCD1, and SCD2 structures are ready."
)
